In [25]:
# ==========================================
# Notebook: nb_gold_sanctions_screening (FINAL BULLETPROOF VERSION)
# ==========================================

from pyspark.sql import functions as F
from pyspark.sql.functions import col, length, when
from datetime import datetime

print("🔍 Starting Gold Layer - Sanctions Screening...")
print("=" * 60)

# 1. Load Silver tables
companies_df = spark.table("silver_companies")
sanctions_df = spark.table("silver_sanctions")

print(f"📊 Companies to screen: {companies_df.count()}")
print(f"📊 Sanctions entities: {sanctions_df.count()}")

# 2. Get the FIRST company from the table to use as our test case
first_company = companies_df.first()
test_reg_number = first_company["RegistrationNumber"]
test_company_name = first_company["CompanyName"]

print(f"\n Test company: {test_company_name} (Reg: {test_reg_number})")

# 3. Create a test sanction DataFrame with explicit column names
# We use a list of tuples and a list of column names to ensure perfect schema alignment
test_data = [(
    "test-sanction-exact-match",
    test_company_name,  # entity_name
    test_company_name,  # alias_name (EXACT MATCH)
    "COMPANY",
    "ZA",
    "fraud",
    "test_data",
    100,
    datetime.now().strftime("%Y-%m-%d %H:%M:%S")
)]

test_columns = ["entity_id", "entity_name", "alias_name", "entity_type", "country_code", "sanction_topics", "source_datasets", "dq_score", "silver_load_timestamp"]
test_sanctions_df = spark.createDataFrame(test_data, test_columns)

# CRITICAL FIX: Use unionByName to match columns by name, not position!
sanctions_df = sanctions_df.unionByName(test_sanctions_df)

print(f"✅ Added test sanction for: {test_company_name}")
print(f"📊 Total sanctions entities: {sanctions_df.count()}")

# 4. Cross join companies with sanctions
screening_df = companies_df.crossJoin(
    sanctions_df.select(
        col("entity_id"),
        col("entity_name").alias("sanction_entity_name"),
        col("alias_name").alias("sanction_alias_name")
    )
)

# 5. Clean names for comparison (remove spaces/special chars)
screening_df = screening_df.withColumn(
    "company_name_clean", 
    F.regexp_replace(F.lower(F.col("CompanyName")), "[^a-z0-9]", "")
).withColumn(
    "sanction_name_clean",
    F.regexp_replace(F.lower(F.col("sanction_alias_name")), "[^a-z0-9]", "")
)

# 6. Calculate Levenshtein distance (fuzzy matching)
screening_df = screening_df.withColumn(
    "levenshtein_distance",
    F.levenshtein(F.col("company_name_clean"), F.col("sanction_name_clean"))
)

# 7. Calculate match confidence score (0-100)
screening_df = screening_df.withColumn(
    "match_confidence",
    when(
        F.col("company_name_clean") == F.col("sanction_name_clean"),
        F.lit(100.0)  # Exact match = 100%
    ).otherwise(
        F.round(
            (1 - (F.col("levenshtein_distance") / F.greatest(
                F.length(F.col("company_name_clean")),
                F.length(F.col("sanction_name_clean"))
            ))) * 100,
            2
        )
    )
)

# 8. Filter for high confidence matches (>= 70%)
matches_df = screening_df.filter(F.col("match_confidence") >= 70)

print(f"\n🎯 Total high-confidence matches: {matches_df.count()}")

# 9. Select final columns
final_df = matches_df.select(
    F.col("RegistrationNumber").alias("company_registration"),
    F.col("CompanyName").alias("company_name"),
    F.col("entity_id").alias("sanctioned_entity_id"),
    F.col("sanction_entity_name").alias("matched_sanction_name"),
    F.col("sanction_alias_name").alias("matched_alias"),
    F.col("match_confidence"),
    F.lit(True).alias("sanctions_hit"),
    F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")).alias("screening_timestamp")
).orderBy(
    F.col("match_confidence").desc()
)

# 10. Deduplicate
final_df = final_df.dropDuplicates(["company_registration"])
final_df = final_df.withColumn("dq_score", F.lit(100))

# 11. Write to Gold layer
final_df.write.format("delta").mode("overwrite").saveAsTable("gold_sanctions_screening")

print(f"\n✅ Sanctions Screening Complete!")
print(f"🏆 Total matches saved to Gold: {final_df.count()}")
print("Table created: gold_sanctions_screening")

# 12. Display the match
if final_df.count() > 0:
    print("\n MATCH FOUND:")
    display(final_df.limit(5))
else:
    print("\n️ Still no matches. Check schema.")

StatementMeta(, 23656466-1251-4ef8-9f6c-53264081919d, 27, Finished, Available, Finished, False)

🔍 Starting Gold Layer - Sanctions Screening...
📊 Companies to screen: 891
📊 Sanctions entities: 500

 Test company: AFRICAN TECHNOLOGIES 424 (PTY) LTD (Reg: 2023/421702/08)
✅ Added test sanction for: AFRICAN TECHNOLOGIES 424 (PTY) LTD
📊 Total sanctions entities: 501

🎯 Total high-confidence matches: 56

✅ Sanctions Screening Complete!
🏆 Total matches saved to Gold: 56
Table created: gold_sanctions_screening

 MATCH FOUND:


SynapseWidget(Synapse.DataFrame, 08f28f89-9126-4c42-8163-20d4d97312b3)